# 4. ICA as a spatial filter: visual and motor components

**Idea.** ICA splits each participant's EEG into components. Each component is a fixed weighting of the electrodes, i.e. a **spatial filter**: its time course is one clean "virtual channel". So far ICA was used to *remove* artifact components. Here it is used to *keep* the brain components we care about (the game has no sound, so no auditory components):

- a **visual component** (alpha, 8–12 Hz);
- a **motor component** (mu, 8–12 Hz, and/or beta, 13–30 Hz).

**No automatic scoring.** The notebook saves one overview PNG with the scalp maps of **all kept components** (the property plot of every component, with time course and spectrum, is already saved by `01_ica` in `<sub>/components/`). The choice is made by eye, with the supervisor, and written into `SPATIAL_FILTER_CHOICE` in `config.py`. Only then are the envelopes computed.

**Steps:** 1. load clean data + ICA → 2. one overview PNG with the scalp maps of all kept components → 3. envelopes of the chosen components (after the review) → 4. save.

Bands are in `config.py` (`SPATIAL_FILTER_BANDS`).

In [ ]:
import importlib
import os

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config as cfg
%matplotlib inline

## 0. Participant

In [ ]:
# execute_notebooks.py sets RT_SUBJECT
SUBJECT = os.environ.get("RT_SUBJECT", "sub-001")
SESSION = cfg.session_of(SUBJECT)
OUT_DIR = cfg.out_dir(SUBJECT)
REVIEW_DIR = cfg.OUT_ROOT / "spatial_filter_review" / SESSION  # one folder per session for the review
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
print(SUBJECT, f"({SESSION})", "->", REVIEW_DIR)

## 1. Load the clean data and the ICA

In [ ]:
raw = mne.io.read_raw(OUT_DIR / f"{SUBJECT}_clean_raw.fif", preload=True)
ica = mne.preprocessing.read_ica(OUT_DIR / f"{SUBJECT}_ica.fif")
removed = pd.read_csv(OUT_DIR / f"{SUBJECT}_badcomps.csv")["component"].astype(int).tolist()
kept = [c for c in range(ica.n_components_) if c not in removed]

sources = ica.get_sources(raw)
print(f"{len(kept)} kept components; removed as artifacts: {removed}")

## 2. Overview: the scalp maps of all kept components in one PNG

Only the maps (no spectra), all kept components of this participant side by side, so they are easy to compare. Saved as `spatial_filter_review/<session>/<subject>_all_maps.png`.

In [ ]:
figure = ica.plot_components(picks=kept, ncols=6, show=False)  # with picks given, MNE puts all maps in one figure
figure = figure[0] if isinstance(figure, list) else figure
figure.suptitle(f"{SUBJECT}: scalp maps of the kept components", fontsize=11)
figure.savefig(REVIEW_DIR / f"{SUBJECT}_all_maps.png", dpi=100, bbox_inches="tight")
plt.close(figure)

## 3. Amplitude envelopes of the chosen components (after the review)

Write the choice into `SPATIAL_FILTER_CHOICE` in `config.py` as `{area: (component, band)}`, e.g. `"sub-001": {"visual": (5, "alpha"), "motor": (8, "beta")}`, then run from here. Without a choice this step and the next do nothing.

Each chosen component is band-passed, then its Hilbert envelope is taken on the **continuous** signal (no edge artifacts per trial). Components have an arbitrary sign and scale; the envelope removes the sign, and cRQA will use standardized envelopes.

In [ ]:
importlib.reload(cfg)  # pick up changes made to config.py
CHOSEN = cfg.SPATIAL_FILTER_CHOICE.get(SUBJECT, {})
if not CHOSEN:
    print(f"{SUBJECT}: no choice in SPATIAL_FILTER_CHOICE yet; envelopes skipped")

envelopes = {}
for area, (comp, band) in CHOSEN.items():
    x = sources.copy().pick([f"ICA{comp:03d}"])
    x.filter(*cfg.SPATIAL_FILTER_BANDS[area][band], picks="all")  # components are 'misc' channels: picks needed
    x.apply_hilbert(envelope=True, picks="all")
    envelopes[area] = x

if envelopes:
    fig, ax = plt.subplots(figsize=(10, 3))
    t0 = 60
    for area, x in envelopes.items():
        sf = x.info["sfreq"]
        seg = x.get_data(start=int(t0 * sf), stop=int((t0 + 10) * sf))[0]
        comp, band = CHOSEN[area]
        ax.plot(np.linspace(t0, t0 + 10, seg.size), seg / seg.std(), label=f"{area}: ICA{comp:03d}, {band}")
    ax.set(xlabel="time (s)", ylabel="envelope (z)"); ax.legend()

## 4. Save

One file per area, with all events (for cutting trials later), and the choice for the Methods.

In [ ]:
for area, x in envelopes.items():
    comp, band = CHOSEN[area]
    x.save(OUT_DIR / f"{SUBJECT}_{area}_{band}_envelope_raw.fif", overwrite=True)

if CHOSEN:
    choice = pd.DataFrame([{"area": area, "component": comp, "band": band} for area, (comp, band) in CHOSEN.items()])
    choice.to_csv(OUT_DIR / f"{SUBJECT}_spatial_filter.csv", index=False)
    choice

## Questions for Anna

1. Motor envelope: mu, beta, or both?
2. One motor component, or one per side (left and right hand)?
3. Choose components per participant by eye, or match them across participants with CORRMAP (a "visual" and a "motor" template), as for the artifacts? Which criteria should the choice by eye use?
4. SSD (spatio-spectral decomposition; Nikulin et al. 2011, Dähne et al. 2014) instead of ICA components, or as a check?
5. Compare with the simple alternative (average over ROI electrodes) as a check?